# Neural CFR+ low-fitting screen

This screen follows the result that `24 regret / 12 strategy` optimizer steps substantially outperformed the previous `100 / 50` reference per unit neural-training time.

It tests the lower fitting boundary, the regret-to-strategy fitting ratio, and whether additional fresh traversals improve the current winner. Every configuration receives the same measured neural-training budget. Exact evaluation runs separately and is excluded from that budget.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.eval.neural_evaluators import (
    AsyncExactExploitabilityEvaluator,
    ScheduledEvaluation,
)
from liars_poker.training.neural_runs import run_neural_cfr_plus

assert torch.cuda.is_available(), 'This screen is intended for a CUDA machine.'
device = 'cuda'
torch.set_float32_matmul_precision('high')
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

SEED = 7
MINUTES_PER_CONFIGURATION = 8
EVALUATE_EVERY_MINUTES = 2
SAVE_CHECKPOINTS = False
DELETE_EVALUATION_SNAPSHOTS = True

CONFIGURATIONS = {
    'control: 1024 traversals; 24 / 12': {
        'traversals_per_player': 1024,
        'regret_train_steps': 24,
        'strategy_train_steps': 12,
    },
    'half fitting: 1024 traversals; 12 / 6': {
        'traversals_per_player': 1024,
        'regret_train_steps': 12,
        'strategy_train_steps': 6,
    },
    'strategy lighter: 1024 traversals; 24 / 6': {
        'traversals_per_player': 1024,
        'regret_train_steps': 24,
        'strategy_train_steps': 6,
    },
    'equal low: 1024 traversals; 12 / 12': {
        'traversals_per_player': 1024,
        'regret_train_steps': 12,
        'strategy_train_steps': 12,
    },
    'more data: 2048 traversals; 24 / 12': {
        'traversals_per_player': 2048,
        'regret_train_steps': 24,
        'strategy_train_steps': 12,
    },
}

BASE_TRAINER_KWARGS = {
    'regret_hidden_sizes': (2048, 2048),
    'strategy_hidden_sizes': (512, 512),
    'device': device,
    'seed': SEED,
    'regret_buffer_capacity': 500_000,
    'strategy_buffer_capacity': 2_000_000,
    'learning_rate': 1e-3,
    'batch_size': 4096,
    'regret_positive_weight': 0.5,
    'strategy_weighting': 'linear',
    'validation_fraction': 0.02,
    'validation_buffer_capacity': 20_000,
    'traversal_backend': 'gpu_native',
    'traversal_batch_size': 1024,
    'device_replay': True,
    'fused_optimizer': True,
    'amp_dtype': None,
    'compile_models': False,
}

screen_id = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
SCREEN_DIR = (
    REPO_ROOT / 'artifacts' / 'neural_method_screens'
    / f'cfr_plus_low_fit_{spec.to_short_str()}___{screen_id}'
)
SCREEN_DIR.mkdir(parents=True, exist_ok=True)
print('screen directory:', SCREEN_DIR)
print('planned measured GPU-training minutes:', len(CONFIGURATIONS) * MINUTES_PER_CONFIGURATION)

In [ ]:
def safe_name(name):
    return ''.join(ch if ch.isalnum() else '_' for ch in name).strip('_').lower()


def exact_schedule():
    return ScheduledEvaluation(
        name='exact_exploitability',
        evaluator=AsyncExactExploitabilityEvaluator(
            max_workers=1,
            compile_batch_size=65_536,
        ),
        every_minutes=EVALUATE_EVERY_MINUTES,
        run_at_end=True,
    )


def mean_validation(metrics, section, metric):
    values = [row[metric] for row in metrics[section] if row.get('records', 0)]
    return float(np.mean(values)) if values else np.nan


def normalized_auc(frame, value_col='exploitability'):
    ordered = frame.sort_values('measured_training_s')
    x = ordered['measured_training_s'].to_numpy(dtype=float)
    y = ordered[value_col].to_numpy(dtype=float)
    if len(x) < 2 or x[-1] <= x[0]:
        return np.nan
    return float(np.trapezoid(y, x) / (x[-1] - x[0]))

In [ ]:
completed_runs = {}


def run_configuration(name, *, overwrite=False):
    if name not in CONFIGURATIONS:
        raise KeyError(f'Unknown configuration: {name}')
    if name in completed_runs and not overwrite:
        print('Already loaded:', name)
        return completed_runs[name]['evaluations']

    config = CONFIGURATIONS[name]
    run_dir = SCREEN_DIR / safe_name(name)
    training_cache = run_dir / 'screen_training.pkl'
    evaluation_cache = run_dir / 'screen_evaluations.csv'
    diagnostic_cache = run_dir / 'screen_diagnostics.csv'

    if overwrite and run_dir.exists():
        shutil.rmtree(run_dir)
    elif all(path.exists() for path in (
        training_cache, evaluation_cache, diagnostic_cache,
    )):
        training = pd.read_pickle(training_cache)
        evaluations = pd.read_csv(evaluation_cache)
        diagnostic = pd.read_csv(diagnostic_cache).iloc[0].to_dict()
        completed_runs[name] = {
            'training': training,
            'evaluations': evaluations,
            'diagnostic': diagnostic,
        }
        print('Loaded completed run:', run_dir)
        return evaluations
    elif run_dir.exists() and any(run_dir.iterdir()):
        raise FileExistsError(
            f'{run_dir} already contains an incomplete or unprocessed run. '
            'Inspect it, or call run_configuration(name, overwrite=True).'
        )

    print(f'\n=== {name} ===')
    trainer_kwargs = {
        **BASE_TRAINER_KWARGS,
        'regret_train_steps': config['regret_train_steps'],
        'strategy_train_steps': config['strategy_train_steps'],
    }
    result = run_neural_cfr_plus(
        spec,
        minutes=MINUTES_PER_CONFIGURATION,
        traversals_per_player=config['traversals_per_player'],
        trainer_kwargs=trainer_kwargs,
        evaluations=[exact_schedule()],
        run_dir=run_dir,
        save_checkpoint=SAVE_CHECKPOINTS,
        wait_for_evaluations=True,
        debug=False,
    )

    training = pd.DataFrame(result.training_records)
    training['configuration'] = name
    training['traversals_per_player'] = config['traversals_per_player']
    training['regret_train_steps'] = config['regret_train_steps']
    training['strategy_train_steps'] = config['strategy_train_steps']

    evaluations = pd.DataFrame(result.evaluation_records)
    evaluations = evaluations[
        evaluations['evaluator'] == 'exact_exploitability'
    ].copy()
    evaluations['configuration'] = name
    evaluations['traversals_per_player'] = config['traversals_per_player']
    evaluations['regret_train_steps'] = config['regret_train_steps']
    evaluations['strategy_train_steps'] = config['strategy_train_steps']
    evaluations = evaluations.sort_values('measured_training_s').drop_duplicates(
        subset='iteration', keep='last'
    )

    validation = result.trainer.validation_metrics()
    diagnostic = {
        'configuration': name,
        'final regret MSE': mean_validation(validation, 'regret', 'mse'),
        'final regret strategy TV': mean_validation(validation, 'regret', 'strategy_tv'),
        'final average-network TV': mean_validation(validation, 'strategy', 'strategy_tv'),
        'run directory': str(result.run_dir),
    }

    training.to_pickle(training_cache)
    evaluations.to_csv(evaluation_cache, index=False)
    pd.DataFrame([diagnostic]).to_csv(diagnostic_cache, index=False)
    completed_runs[name] = {
        'training': training,
        'evaluations': evaluations,
        'diagnostic': diagnostic,
    }

    if DELETE_EVALUATION_SNAPSHOTS:
        snapshots = run_dir / 'evaluations' / 'exact_exploitability' / 'snapshots'
        if snapshots.exists():
            shutil.rmtree(snapshots)

    print(
        f"iterations={result.trainer.iteration} "
        f"train={result.measured_training_s / 60.0:.2f}m "
        f"evaluations={len(evaluations)}"
    )
    del result
    gc.collect()
    torch.cuda.empty_cache()
    return evaluations


def combined_results():
    if not completed_runs:
        raise RuntimeError('Run at least one configuration first.')
    training = pd.concat(
        [run['training'] for run in completed_runs.values()],
        ignore_index=True,
    )
    evaluations = pd.concat(
        [run['evaluations'] for run in completed_runs.values()],
        ignore_index=True,
    )
    diagnostics = pd.DataFrame(
        [run['diagnostic'] for run in completed_runs.values()]
    ).set_index('configuration')
    training.to_pickle(SCREEN_DIR / 'training_results.pkl')
    evaluations.to_csv(SCREEN_DIR / 'evaluation_results.csv', index=False)
    diagnostics.to_csv(SCREEN_DIR / 'final_diagnostics.csv')
    return training, evaluations, diagnostics

In [ ]:
run_configuration('control: 1024 traversals; 24 / 12')

In [ ]:
run_configuration('half fitting: 1024 traversals; 12 / 6')

In [ ]:
run_configuration('strategy lighter: 1024 traversals; 24 / 6')

In [ ]:
run_configuration('equal low: 1024 traversals; 12 / 12')

In [ ]:
run_configuration('more data: 2048 traversals; 24 / 12')

In [ ]:
training_results, evaluation_results, diagnostics = combined_results()
summary_rows = []

for name, evaluations in evaluation_results.groupby('configuration', sort=False):
    evaluations = evaluations.sort_values('measured_training_s')
    training = training_results[training_results['configuration'] == name]
    final = evaluations.iloc[-1]
    timing = pd.json_normalize(training['timing'])
    config = CONFIGURATIONS[name]

    summary_rows.append({
        'configuration': name,
        'traversals': config['traversals_per_player'],
        'regret steps': config['regret_train_steps'],
        'strategy steps': config['strategy_train_steps'],
        'iterations completed': int(training['iteration'].iloc[-1]),
        'iterations / measured min': (
            training['iteration'].iloc[-1]
            / (training['measured_training_s'].iloc[-1] / 60.0)
        ),
        'final exploitability': final['exploitability'],
        'best exploitability': evaluations['exploitability'].min(),
        'normalized AUC': normalized_auc(evaluations),
        'mean traversal s': timing['traversal_s'].mean(),
        'mean regret fit s': timing['regret_training_s'].mean(),
        'mean strategy fit s': timing['strategy_training_s'].mean(),
    })

summary = pd.DataFrame(summary_rows).set_index('configuration')
summary = summary.join(diagnostics.drop(columns='run directory'))
summary = summary.sort_values('normalized AUC')
display(summary.style.format(precision=6))
summary.to_csv(SCREEN_DIR / 'summary.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

for name, frame in evaluation_results.groupby('configuration', sort=False):
    frame = frame.sort_values('measured_training_s')
    axes[0].plot(
        frame['measured_training_s'] / 60.0,
        frame['exploitability'],
        marker='o',
        label=name,
    )
    axes[1].plot(
        frame['iteration'],
        frame['exploitability'],
        marker='o',
        label=name,
    )

for ax, title, xlabel in [
    (axes[0], 'Exact exploitability by training time', 'Measured neural-training minutes'),
    (axes[1], 'Exact exploitability by CFR+ iteration', 'CFR+ iteration'),
]:
    ax.set(title=title, xlabel=xlabel, ylabel='Exploitability')
    ax.set_yscale('log')
    ax.grid(True, which='both', alpha=0.3)

axes[0].legend(fontsize=8)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

timing_summary = summary[[
    'mean traversal s', 'mean regret fit s', 'mean strategy fit s'
]].copy()
timing_summary.columns = ['traversal', 'regret fit', 'strategy fit']
timing_summary.plot.bar(stacked=True, ax=axes[0])
axes[0].set(
    title='Mean measured iteration cost',
    xlabel='',
    ylabel='Seconds per iteration',
)
axes[0].tick_params(axis='x', labelrotation=25)
axes[0].grid(axis='y', alpha=0.3)

axes[1].scatter(
    summary['iterations / measured min'],
    summary['final exploitability'],
    s=70,
)
for name, row in summary.iterrows():
    axes[1].annotate(
        name,
        (row['iterations / measured min'], row['final exploitability']),
        xytext=(5, 4),
        textcoords='offset points',
        fontsize=8,
    )
axes[1].set(
    title='Iteration throughput versus final quality',
    xlabel='Iterations per measured minute',
    ylabel='Final exact exploitability',
)
axes[1].grid(alpha=0.3)
fig.tight_layout()

## Selection rule

Prefer normalized exploitability AUC and final exact exploitability over current-strategy diagnostics. A lower-fitting configuration is credible if its validation TV remains controlled while it completes enough additional CFR+ iterations to improve the average strategy.

If `2048 traversals / 24 / 12` wins, the next reference run should use it. If one of the 12-step regret variants wins, run that winner and the `24 / 12` control for a longer horizon before reducing fitting further.